In [ ]:
import json
from pathlib import Path
import pickle
from sentence_transformers import SentenceTransformer
import ipywidgets as widgets
from IPython.display import display
from dotenv import load_dotenv

project_root = Path(Path.cwd()).resolve().parent
env_path = project_root / '.env'

load_dotenv(dotenv_path=env_path, override=True)

labels_dir = project_root / 'data' / 'potential_labels'
model_path = project_root / 'models' / 'embeddings.pkl'

# Load a small pre-trained embedding model (open local model)
model = SentenceTransformer('all-MiniLM-L6-v2')  # open-source embedding model

# Ensure the labels directory exists
if not labels_dir.exists():
    print(f"Directory {labels_dir} does not exist. Please create it and add JSON files.")
    exit(1)

# Collect JSON files
files = sorted([p for p in labels_dir.iterdir() if p.is_file() and p.suffix == '.json'])

progress = widgets.IntProgress(value=0, min=0, max=len(files), description='Progress:')
pct = widgets.HTML(value="0%")
display(widgets.HBox([progress, pct]))
embedding_db = []  # list to store {"id", "name", "vector"} for each recipe
for idx, fp in enumerate(files):
    try:
        text = fp.read_text(encoding='utf-8')
        obj = json.loads(text)   
    except Exception as e:
        # Log and continue on parse/read errors
        print(f'Skipping {fp.name}: {e}')
        continue

    progress.value = idx + 1
    pct.value = f"{int(100 * (idx + 1) / len(files))}%"

    # add source filename so downstream steps know which file produced this record
    if isinstance(obj, dict) and obj.get("title") and obj.get("ingredients"):
        title_instructions_file = {
            
        }
        text = f"{obj["title"]}: " + ", ".join(obj["ingredients"])
        vector = model.encode(text)            # get embedding vector (e.g., numpy array)
        embedding_db.append({
            "title" : obj["title"],
            "ingredients": obj["ingredients"],
            "source_file": fp.name,
            "vector": vector
        })
        
progress.value = len(files)
pct.value = "100%"

pickle.dump(embedding_db, open(model_path, 'wb'))
print(f"Stored {len(embedding_db)} recipe embeddings in the database.")

In [ ]:
embedding_db = pickle.load(open(model_path, 'rb'))

In [ ]:
import numpy as np

def find_best_matches(query_ingredients, top_n: int = 10):
    """
    Return the top_n matching recipe entries from embedding_db for the given
    query_ingredients list. Returns a list of tuples: (entry_dict, score).
    """
    # Embed the query ingredients list into a vector
    query_text = ", ".join(query_ingredients)
    q_vec = model.encode(query_text)

    # Stack all stored vectors into a matrix (N x D)
    mat = np.vstack([entry["vector"] for entry in embedding_db])

    # Compute cosine similarities in a vectorized way
    q_norm = np.linalg.norm(q_vec)
    mat_norms = np.linalg.norm(mat, axis=1)
    denom = mat_norms * (q_norm if q_norm != 0 else 1e-12)

    # avoid division by zero
    denom[denom == 0] = 1e-12
    sims = (mat @ q_vec) / denom

    # Get top_n indices (highest similarity)
    k = min(int(top_n), sims.size)
    top_idx = np.argsort(sims)[::-1][:k]

    # Build results: (entry, score)
    return [(embedding_db[i], float(sims[i])) for i in top_idx]
    


# Example query
user_ingredients = ["cheese", "bread", "mustard", "pickle"]
matches = find_best_matches(user_ingredients)
for entry, score in matches:
    print(f"{entry['title']} [{entry['source_file']}] — {score * 100:.2f}%")

In [ ]:
# Attempt to extract JSON object from the assistant text
import json
from json import JSONDecoder
from typing import Any

def _extract_json_from_text(text: str):
    # quick whole-text parse
    try:
        return json.loads(text)
    except Exception:
        pass
    dec = JSONDecoder()
    for start in ("{", "["):
        idx = text.find(start)
        while idx != -1:
            try:
                obj, end = dec.raw_decode(text[idx:])
                return obj
            except json.JSONDecodeError:
                idx = text.find(start, idx + 1)
    return None

def _verify_file_and_title(file_name, recipe_name, labels_dir):
    candidate_path = labels_dir / file_name
    if candidate_path.exists():
        try:
            rf = json.loads(candidate_path.read_text(encoding="utf-8"))
            file_title = (rf.get("title") or rf.get("recipe_name") or "").strip()
            if recipe_name and file_title and file_title.lower() == recipe_name.strip().lower():
                return file_name  # good match
            else:
                # mismatch: try to find a file whose title matches the returned recipe_name
                if recipe_name:
                    for p in files:
                        try:
                            o = json.loads(p.read_text(encoding="utf-8"))
                            t = (o.get("title") or o.get("recipe_name") or "").strip()
                            if t and t.lower() == recipe_name.strip().lower():
                                return p.name
                        except Exception:
                            continue

                    print(f"Warning: file {file_name} title '{file_title}' did not match recipe_name '{recipe_name}', no alternative found.")
                else:
                    print(f"Warning: file {file_name} title '{file_title}' did not match empty recipe_name.")
        except Exception as e:
            print(f"Warning: failed to read/parse {candidate_path}: {e}")
    else:
        print(f"Warning: referenced file {file_name} not found under {labels_dir}")

    return None


def _get_total_tokens(resp: Any) -> int | None:
    """
    Accepts a dict or JSON string (LLM response) and returns total token count if available.
    """
    if isinstance(resp, str):
        try:
            resp = json.loads(resp)
        except Exception:
            return None
    if not isinstance(resp, dict):
        return None

    # preferred location
    usage = resp.get("usage")
    if isinstance(usage, dict) and "total_tokens" in usage:
        return int(usage["total_tokens"])

    # fallback: sum prompt + completion if present
    pt = usage.get("prompt_tokens") if isinstance(usage, dict) else resp.get("prompt_tokens")
    ct = usage.get("completion_tokens") if isinstance(usage, dict) else resp.get("completion_tokens")
    try:
        if pt is not None and ct is not None:
            return int(pt) + int(ct)
    except Exception:
        pass

    return None

In [ ]:
import requests
import os
import time

useOpenAI = True

user_ingredients = ["carrots", "edamame", "corn", "pork"]  # user has these ingredients

# Grab the top 25 matches from the embedding DB
top_matches = find_best_matches(user_ingredients, top_n=25)

# Construct a prompt with the query and top matches to send to the LLM
candidate_list_str = ""
for i, (recipe, score) in enumerate(top_matches, start=1):
    ingr_list = ", ".join(recipe["ingredients"])
    candidate_list_str += f'{{ "recipe_name": "{recipe['title']}", "file_name": "{recipe['source_file']}", "ingredients":  "{ingr_list}" }}'

recipe_style = "recipe that is good for lunch"

system_msg = (
    "You are a helpful cooking assistant. A user has certain ingredients, and we have some candidate recipes from a database. "
    "Choose which recipe is the best match for the user's ingredients and give a reason for choosing it."
)

user_msg = (
    f"The user has the following ingredients: {', '.join(user_ingredients)}.\n"
    f"The candidate recipes are:\n{candidate_list_str}\n"
    f"The user wants a recipe that matches '{recipe_style}'\n"
    "Which recipe from the candidate recipes best matches the user's ingredients and has the style that user wants?\n"
    "Respond with the recipe name and file_name and the reason for picking this recipe in JSON. Like this: "
    "{\"recipe_name\": \"Tomato Soup\", \"file_name\": \"recipe_00031.json\", \"reason\": \"Because soup is good food\"}"
)

# if we have an OPENAI_API_KEY environment variable and we have set useOpenAI=True, use OpenAI's API
if useOpenAI and os.getenv("OPENAI_API_KEY"):
    API_KEY = os.getenv("OPENAI_API_KEY")
    api_url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    #model_name =  "gpt-5-nano"
    model_name = "gpt-5-nano"
else:
    # Use Ollama local model server
    api_url = "http://127.0.0.1:11434/v1/chat/completions"
    headers = {
        "Content-Type": "application/json"
    }
    # model_name = "phi4-mini"
    # model_name = "qwen3:0.6b"
    # model_name = "deepseek-r1:1.5b"
    model_name = "tinyllama"
    # model_name = "gemma3:1b"

data = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]
}

if not model_name.startswith("gpt-5"):
    data["temperature"] = 0.3
else:
    data["reasoning_effort"] = "minimal"

# Send the HTTP request to OpenAI
start = time.perf_counter()
response = requests.post(api_url, headers=headers, json=data)
time_elapsed = time.perf_counter() - start
response_json = response.json()

if response_json.get("error") or response.status_code != 200:
    print(f"Error: {response.status_code} - {response.text}")
else:
    # Extract the assistant's answer (recipe title and id)
    total_tokens = _get_total_tokens(response_json)  # just to demonstrate usage; ignore return value
    best_recipe_answer = response_json["choices"][0]["message"]["content"].strip()
    print(f"{model_name}'s answer took {total_tokens} tokens and {time_elapsed:.3f}s:\n{best_recipe_answer}")

    parsed = _extract_json_from_text(best_recipe_answer)
    chosen_file = None

    if isinstance(parsed, dict):
        file_name = parsed.get("file_name") or parsed.get("file")
        recipe_name = parsed.get("recipe_name") or parsed.get("recipe") or parsed.get("title")
        if file_name:
            chosen_file = _verify_file_and_title(file_name, recipe_name, labels_dir)
        else:
            # no file_name provided; try to resolve by recipe_name
            print(f"No file_name given to '{recipe_name}'.")

    else:
        print("No JSON found in model response; skipping filename/title validation.")

    # chosen_file is either the validated filename (string) or None
    if chosen_file:
        print(f"Final chosen file for the model's pick: {chosen_file}")
    else:
        print("No validated file selected for the model's pick.")


